In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch12-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "12-04-bayesian-optimization.py"

# 用 ESOL 重播期望改善量選點

執行分析格會寫出 Python 程式，最後一格執行並顯示圖。試做練習時，可將程式接在分析格末尾再執行這兩格。

In [ ]:
%%writefile "{script}"
from collections import defaultdict
from pathlib import Path
import os

import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.preprocessing import StandardScaler

# ESOL: 1,128 measurements of log10(S / (mol/L)), DOI 10.1021/ci034243x.
_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
records = defaultdict(list)
for smi, value, weight in zip(data.ids, data.y[:, 0], data.w[:, 0]):
    if weight > 0 and np.isfinite(value):
        records[Chem.MolToSmiles(Chem.MolFromSmiles(smi))].append(float(value))
smiles = np.array(sorted(records))

def reveal(i):
    return np.mean(records[smiles[i]])

def descriptors(smi):
    mol = Chem.MolFromSmiles(smi)
    return [Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol)]

def fit_gp(indices):
    model = GaussianProcessRegressor(kernel=Matern(length_scale=2, nu=2.5),
                                     alpha=0.1, optimizer=None, normalize_y=True)
    return model.fit(X[indices], [reveal(i) for i in indices])

out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
# 抽取候選時只使用結構。全庫結構已知，可先用這些描述符決定各欄尺度。
library = np.sort(np.random.default_rng(1204).choice(len(smiles), 128, replace=False))
smiles = smiles[library]
X = StandardScaler().fit_transform([descriptors(s) for s in smiles])

def expected_improvement(mu, sigma, best):
    gain = mu - best
    z = np.divide(gain, sigma, out=np.zeros_like(gain), where=sigma > 0)
    return np.where(sigma > 0, gain * norm.cdf(z) + sigma * norm.pdf(z),
                    np.maximum(gain, 0.0))

def campaign(rule, seed, steps=12, kappa=1.5):
    rng = np.random.default_rng(seed)
    seen = list(rng.choice(len(smiles), 8, replace=False))
    best = max(reveal(i) for i in seen)
    trace = [best]
    journal = []
    for _ in range(steps):
        left = np.setdiff1d(np.arange(len(smiles)), seen)
        model = fit_gp(seen)
        mu, sigma = model.predict(X[left], return_std=True)
        if rule == "ei":
            score = expected_improvement(mu, sigma, best)
        elif rule == "ucb":
            score = mu + kappa * sigma
        elif rule == "random":
            score = rng.random(len(left))
        else:
            raise ValueError(rule)
        j = np.argmax(score)
        i = left[j]
        value = reveal(i)
        journal.append((i, mu[j], sigma[j], score[j], value))
        seen.append(i)
        best = max(best, value)
        trace.append(best)
    return np.array(trace), journal
trace, journal = campaign("ei", 42)
random_trace, _ = campaign("random", 42)
print("library, initial best:", len(smiles), trace[0])
for step, (i, mu, sigma, score, value) in enumerate(journal, 1):
    print(step, i, smiles[i], np.round([mu, sigma, score, value], 4))
print("random record:", random_trace)
fig, ax = plt.subplots(figsize=(7.2, 4.1), layout="constrained")
budget = np.arange(8, 21)
ax.step(budget, random_trace, where="post", color="#697789", ls="--", label="Random best")
ax.step(budget, trace, where="post", color="#007875", lw=2.2, label="EI best")
ax.scatter(budget[1:], [row[-1] for row in journal], s=27, color="#c16728", label="New EI measurement")
ax.set(xlabel="Structures measured (including 8 initial)", ylabel="Measured log S",
       title="Finding soluble molecules in a fixed ESOL library", xticks=[8, 10, 12, 14, 16, 18, 20])
ax.legend()
fig.savefig(out / "12-04-bayesian-optimization.png", dpi=170)


In [ ]:
from IPython.display import Image, display

import os
environment = dict(os.environ, VECLIB_MAXIMUM_THREADS="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MPLBACKEND="Agg")
subprocess.run([python, str(script)], check=True, env=environment)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "12-04-bayesian-optimization.png")))